# 01 - Venue Data & Schema Inspection

This notebook inspects the raw-to-interim extracted High-Volume FHV records for January 2024.
It validates:
- Schema and column data types
- Null values
- Pickup location distributions across Barclays Center, Citi Field, and designated spillover zones.

In [ ]:
from pathlib import Path
import pandas as pd

# Define path to the interim filtered parquet file
DATA_PATH = Path('../data/interim/venue_trips_2024-01.parquet')
if not DATA_PATH.exists():
    # Fallback if executing from root directory
    DATA_PATH = Path('data/interim/venue_trips_2024-01.parquet')

print(f'Loading data from: {DATA_PATH.resolve()}')
df = pd.read_parquet(DATA_PATH)
print(f'Successfully loaded {len(df):,} records.')

: 

### 1. Shape and Schema Overview

In [ ]:
print('Dataset Dimensions (Rows, Columns):', df.shape)
print('\n--- Column Data Types ---')
df.info()

### 2. Missing Value Analysis

In [ ]:
null_summary = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Percentage (%)': (df.isna().sum() / len(df) * 100).round(2)
})
null_summary

### 3. Target Zone Breakdown
- **Barclays Center:** Core = `189`, Spillovers = `[49, 62, 69, 97, 181, 190]`
- **Citi Field:** Core = `253`, Spillover = `[93]`

In [ ]:
zone_labels = {
    189: 'Barclays Core (Prospect Heights)',
    49: 'Barclays Spillover (Clinton Hill)',
    62: 'Barclays Spillover (Crown Heights South)',
    69: 'Barclays Spillover (East Concourse)',
    97: 'Barclays Spillover (Fort Greene)',
    181: 'Barclays Spillover (Park Slope)',
    190: 'Barclays Spillover (Prospect Park)',
    253: 'Citi Field Core (Willets Point)',
    93: 'Citi Field Spillover (Flushing Meadows-Corona Park)'
}

zone_counts = df['PULocationID'].value_counts().sort_index().reset_index()
zone_counts.columns = ['PULocationID', 'Trip_Count']
zone_counts['Zone_Description'] = zone_counts['PULocationID'].map(zone_labels)
zone_counts['Share (%)'] = (zone_counts['Trip_Count'] / len(df) * 100).round(2)
zone_counts[['PULocationID', 'Zone_Description', 'Trip_Count', 'Share (%)']]

### 4. Sample Record Preview

In [ ]:
df.head(10)